# UC3_Product_Intelligence_Assistant

```
=============================================================================
UC3 — PRODUCT INTELLIGENCE ASSISTANT (RAG)
=============================================================================
Reads from : dbx_retail.gold.dim_products
             dbx_retail.gold.dim_vendors
             dbx_retail.gold.mv_slow_moving_products
             dbx_retail.gold.mv_return_rate_by_vendor
Writes to  : dbx_retail.gold.rag_query_history
             dbx_retail.gold.pipeline_run_log
Index path : /Volumes/dbx_retail/gold/rag_artifacts/
Model      : databricks-gpt-oss-20b via Databricks AI Gateway
Embeddings : sentence-transformers/all-MiniLM-L6-v2 (local, no API cost)
=============================================================================
DESIGN NOTES
-------------------------------------------------------
RAG vs Genie: these solve different problems.
  RAG    = semantic questions: "which vendors supply frequently returned products?"
  Genie  = analytical SQL questions: "how many products have return rate > 20%?"

Document construction matters more than retrieval algorithm.
A product document that includes slow_moving flag, vendor name, return rate,
and region availability retrieves correctly for almost any merchandising question.
A document that only has product name and category will miss most queries.

FAISS index is persisted to Databricks Volumes so it survives cluster restarts.
On subsequent runs, the index is only rebuilt if new products were added.
=============================================================================
```

In [0]:
df_dim_customers         = spark.table("globalmart.gold.dim_customers")
df_dim_dates             = spark.table("globalmart.gold.dim_dates")
df_dim_products          = spark.table("globalmart.gold.dim_products")
df_dim_vendors           = spark.table("globalmart.gold.dim_vendors")
df_dq_audit_report       = spark.table("globalmart.gold.dq_audit_report")
df_fact_orders           = spark.table("globalmart.gold.fact_orders")
df_fact_returns          = spark.table("globalmart.gold.fact_returns")
df_fact_sales            = spark.table("globalmart.gold.fact_sales")
df_mv_customer_return_history = spark.table("globalmart.gold.mv_customer_return_history")
df_mv_return_rate_by_vendor   = spark.table("globalmart.gold.mv_return_rate_by_vendor")
df_mv_revenue_by_region      = spark.table("globalmart.gold.mv_revenue_by_region")
df_mv_slow_moving_products   = spark.table("globalmart.gold.mv_slow_moving_products")
df_pipeline_run_log          = spark.table("globalmart.gold.pipeline_run_log")

print("Gold table row counts:")
print(f"  dim_customers                : {df_dim_customers.count()}")
print(f"  dim_dates                    : {df_dim_dates.count()}")
print(f"  dim_products                 : {df_dim_products.count()}")
print(f"  dim_vendors                  : {df_dim_vendors.count()}")
print(f"  dq_audit_report              : {df_dq_audit_report.count()}")
print(f"  fact_orders                  : {df_fact_orders.count()}")
print(f"  fact_returns                 : {df_fact_returns.count()}")
print(f"  fact_sales                   : {df_fact_sales.count()}")
print(f"  mv_customer_return_history   : {df_mv_customer_return_history.count()}")
print(f"  mv_return_rate_by_vendor     : {df_mv_return_rate_by_vendor.count()}")
print(f"  mv_revenue_by_region         : {df_mv_revenue_by_region.count()}")
print(f"  mv_slow_moving_products      : {df_mv_slow_moving_products.count()}")
print(f"  pipeline_run_log             : {df_pipeline_run_log.count()}")

In [0]:
%pip install sentence-transformers faiss-cpu openai
dbutils.library.restartPython()

In [0]:
import os
import json
import time
import numpy as np
import pandas as pd
import faiss
from datetime import datetime
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from openai import OpenAI

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1"
)

MODEL_NAME     = "databricks-gpt-oss-20b"
EMBED_MODEL    = "all-MiniLM-L6-v2"
NOTEBOOK_NAME  = "UC3_Product_Intelligence_Assistant"

# Persistent storage paths — survives cluster restart
INDEX_PATH    = "/Volumes/dbx_retail/gold/rag_artifacts/product_index.faiss"
DOCS_PATH     = "/Volumes/dbx_retail/gold/rag_artifacts/product_docs.json"
META_PATH     = "/Volumes/dbx_retail/gold/rag_artifacts/product_meta.json"
VERSION_PATH  = "/Volumes/dbx_retail/gold/rag_artifacts/index_version.json"

# Create artifact directory if it does not exist
spark.sql("CREATE VOLUME IF NOT EXISTS dbx_retail.gold.rag_artifacts")   

print("Imports complete.")
print(f"Index will be persisted at: {INDEX_PATH}")

In [0]:

def extract_text(response) -> str:
    content = response.choices[0].message.content
    if isinstance(content, list):
        parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(parts).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.1    # low temperature — factual grounded answers only
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"


def validate_llm_output(text: str, min_words: int = 10) -> dict:
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable",
                       "i could not find that information"]
    issues = []
    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(p in text.lower() for p in refusal_phrases[:4]):
        # Note: "i could not find" is a valid RAG response when docs don't contain answer
        issues.append("possible_refusal")
    return {
        "text":         text,
        "quality_flag": "PASS" if not issues else "REVIEW",
        "issues":       ", ".join(issues) if issues else "none"
    }


print("LLM helpers defined.")

In [0]:

df_products = spark.table("dbx_retail.gold.dim_products")
df_vendors  = spark.table("dbx_retail.gold.dim_vendors")

# Load Materialized Views for performance context
# These enrich product documents with business-critical metrics
try:
    df_slow_moving  = spark.table("dbx_retail.gold.mv_slow_moving_products")
    df_vendor_rates = spark.table("dbx_retail.gold.mv_return_rate_by_vendor")
    print("Materialized views loaded.")
except Exception as e:
    print(f"MV load warning: {e}. Proceeding with base tables.")
    df_slow_moving  = spark.createDataFrame([], schema="product_id string, is_slow_moving string")
    df_vendor_rates = spark.createDataFrame([], schema="vendor_id string, return_rate double")

# Enrich products with vendor name and performance data
df_enriched = (
    df_products
    .join(df_vendors.select("vendor_id", "vendor_name"), on="vendor_id", how="left")
    .join(
        df_slow_moving.select("product_id", "is_slow_moving", "region")
                      .withColumnRenamed("region", "slow_region"),
        on="product_id", how="left"
    )
    .join(
        df_vendor_rates.select("vendor_id", "return_rate"),
        on="vendor_id", how="left"
    )
    .fillna({
        "vendor_name":   "Unknown Vendor",
        "is_slow_moving": "No",
        "slow_region":   "N/A",
        "return_rate":    0.0
    })
    .toPandas()
)

print(f"Enriched product records: {len(df_enriched)}")
print(f"Columns available: {list(df_enriched.columns)}")

In [0]:
# Document quality determines retrieval quality.
# Each document includes: product ID, name, category, vendor, price,
# slow-moving flag, region context, vendor return rate.
# Omitting slow_moving flag or vendor name from the document means
# questions about those attributes will retrieve the wrong products.

def row_to_document(row) -> str:
    """
    Converts a product row to a rich natural language document.
    Every attribute that a merchandising team might ask about is included.
    """
    def safe(val, default="N/A"):
        return str(val) if val is not None and str(val) not in ("nan", "None", "") else default

    slow_flag = safe(row.get("is_slow_moving", "No"))
    slow_note = (
        f"This product is flagged as slow-moving in the {safe(row.get('slow_region'))} region."
        if slow_flag.lower() in ("yes", "true", "1") else
        "This product is not flagged as slow-moving."
    )

    vendor_rate = row.get("return_rate", 0)
    try:
        vendor_rate_pct = f"{round(float(vendor_rate) * 100, 1)}%"
    except Exception:
        vendor_rate_pct = "N/A"

    return (
        f"Product {safe(row.get('product_id'))} is a {safe(row.get('product_name'))} "
        f"in the {safe(row.get('category'))} category. "
        f"It is supplied by {safe(row.get('vendor_name'))} (vendor ID: {safe(row.get('vendor_id'))}). "
        f"Listed price: ${safe(row.get('unit_price'))}. "
        f"Available in region: {safe(row.get('region'))}. "
        f"{slow_note} "
        f"The supplier for this product has an overall return rate of {vendor_rate_pct}."
    )

documents = []
metadata  = []

for _, row in df_enriched.iterrows():
    doc = row_to_document(row)
    documents.append(doc)
    metadata.append(row.to_dict())

print(f"Created {len(documents)} product documents.")
print(f"\nSample document:\n{documents[0]}")

In [0]:
# Index is persisted to Volumes. On re-runs, it is loaded rather than rebuilt.
# A version file tracks document count. If product count changed, rebuild.

embed_model = SentenceTransformer(EMBED_MODEL)
print(f"Embedding model loaded: {EMBED_MODEL}")

def build_and_save_index(documents, metadata):
    """Embeds all documents, builds FAISS index, saves to Volumes."""
    print("Building FAISS index from scratch...")
    embeddings = embed_model.encode(documents, show_progress_bar=True)
    dimension  = embeddings.shape[1]
    index      = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings, dtype=np.float32))

    # Save to Volumes for persistence
    faiss.write_index(index, INDEX_PATH)
    with open(DOCS_PATH, "w") as f:
        json.dump(documents, f)
    with open(META_PATH, "w") as f:
        json.dump([{k: str(v) for k, v in m.items()} for m in metadata], f)
    with open(VERSION_PATH, "w") as f:
        json.dump({"doc_count": len(documents), "built_at": datetime.now().isoformat()}, f)

    print(f"FAISS index saved — {index.ntotal} vectors | dimension: {dimension}")
    return index

def load_index():
    """Loads persisted FAISS index and documents from Volumes."""
    index = faiss.read_index(INDEX_PATH)
    with open(DOCS_PATH, "r") as f:
        docs = json.load(f)
    print(f"FAISS index loaded from Volumes — {index.ntotal} vectors.")
    return index, docs

# Check if we need to rebuild
rebuild_needed = True
try:
    with open(VERSION_PATH, "r") as f:
        version_info = json.load(f)
    if version_info.get("doc_count") == len(documents):
        rebuild_needed = False
        print(f"Index is current (doc_count={len(documents)}). Loading from Volumes.")
    else:
        print(f"Product count changed ({version_info.get('doc_count')} -> {len(documents)}). Rebuilding.")
except Exception:
    print("No existing index found. Building fresh.")

if rebuild_needed:
    index = build_and_save_index(documents, metadata)
else:
    index, documents = load_index()

print(f"\nIndex ready — {index.ntotal} vectors indexed.")

In [0]:
# Mirrors AI_Vehicle_Knowledge_Assistant pattern exactly.
# Added: retrieved document transparency (distances + doc text printed).

def rag_query(question: str, top_k: int = 3) -> dict:
    """
    Full RAG pipeline:
    1. Embed the user question
    2. Search FAISS for top_k most similar product documents
    3. Build a grounded prompt with retrieved context
    4. Call LLM — instructed to answer ONLY from retrieved docs
    5. Return answer + retrieved docs + distances for transparency

    If the answer is not in the index, the LLM says so explicitly.
    """
    # Embed question
    q_embedding = np.array(embed_model.encode([question]), dtype=np.float32)

    # FAISS search
    distances, indices = index.search(q_embedding, top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])

    # Build grounded prompt
    prompt = f"""You are a product intelligence assistant for dbx_retail, a national retail chain.
Answer the merchandising team's question using ONLY the product documents below.
If the answer is not in the documents, say exactly:
"I could not find that information in the current product catalog."
Do not invent product names, vendor names, or return rates not present in the documents.

PRODUCT DOCUMENTS:
{context}

QUESTION: {question}

Answer in plain English. Be specific — mention product IDs, names, vendors, regions, and return rates
where relevant."""

    raw_answer = call_llm(
        prompt,
        system_msg="You are a helpful product catalog assistant for dbx_retail retail. "
                   "Only answer based on the documents provided. Never fabricate information."
    )

    return {
        "question":        question,
        "answer":          raw_answer,
        "retrieved_docs":  retrieved_docs,
        "distances":       [round(float(d), 4) for d in distances[0]],
        "top_k":           top_k
    }


print("RAG query function ready.")

In [0]:
# Covers: slow-moving lookup, vendor return rate, region availability,
# category performance, and cross-dimension semantic query.

test_questions = [
    "Which products are slow-moving in the West region?",
    "Which vendors have the highest return rate?",
    "What products are available in the East region?",
    "Show me products in the furniture category.",
    "Which vendor supplies products that are both slow-moving and have a high return rate?"
]

query_log      = []
llm_call_count = 0
review_count   = 0

for question in test_questions:
    result    = rag_query(question, top_k=3)
    validated = validate_llm_output(result["answer"])
    llm_call_count += 1

    if validated["quality_flag"] == "REVIEW":
        review_count += 1

    print("=" * 70)
    print(f"QUESTION  : {question}")
    print("-" * 70)
    print("RETRIEVED DOCUMENTS:")
    for i, (doc, dist) in enumerate(zip(result["retrieved_docs"], result["distances"]), 1):
        print(f"  {i}. [distance={dist}] {doc}")
    print("-" * 70)
    print(f"ANSWER (quality={validated['quality_flag']}):")
    print(validated["text"])
    print("=" * 70 + "\n")

    query_log.append({
        "question":            question,
        "answer":              validated["text"],
        "quality_flag":        validated["quality_flag"],
        "quality_issues":      validated["issues"],
        "retrieved_doc_1":     result["retrieved_docs"][0] if len(result["retrieved_docs"]) > 0 else "",
        "retrieved_doc_2":     result["retrieved_docs"][1] if len(result["retrieved_docs"]) > 1 else "",
        "retrieved_doc_3":     result["retrieved_docs"][2] if len(result["retrieved_docs"]) > 2 else "",
        "similarity_distance": str(result["distances"]),
        "queried_at":          datetime.now().isoformat()
    })

print(f"Queries run: {len(query_log)} | LLM calls: {llm_call_count} | For review: {review_count}")

In [0]:

schema = StructType([
    StructField("question",            StringType(), True),
    StructField("answer",              StringType(), True),
    StructField("quality_flag",        StringType(), True),
    StructField("quality_issues",      StringType(), True),
    StructField("retrieved_doc_1",     StringType(), True),
    StructField("retrieved_doc_2",     StringType(), True),
    StructField("retrieved_doc_3",     StringType(), True),
    StructField("similarity_distance", StringType(), True),
    StructField("queried_at",          StringType(), True)
])

df_query_log = spark.createDataFrame(query_log, schema=schema)

(
    df_query_log.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("dbx_retail.gold.rag_query_history")
)

print("Query history written to dbx_retail.gold.rag_query_history")
display(spark.table("dbx_retail.gold.rag_query_history"))

In [0]:

run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": len(documents),
    "llm_calls_made":    llm_call_count,
    "outputs_flagged":   review_count,
    "status":            "SUCCESS",
    "notes":             f"index_rebuilt={rebuild_needed} | docs_indexed={index.ntotal}"
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("dbx_retail.gold.pipeline_run_log")
)

print(f"Run log written. Status: SUCCESS")


# =============================================================================
# PART B — GENIE SPACE SETUP (Databricks UI — no code required)
# =============================================================================
# This section documents the Genie configuration steps. Execute in the UI.
# =============================================================================
#
# STEP 1 — Create a Genie Space
#   Go to: Databricks workspace > New > Genie Space
#   Name it: "dbx_retail Product Intelligence"
#
# STEP 2 — Add these tables to the Genie Space:
#   dbx_retail.gold.fact_orders
#   dbx_retail.gold.fact_returns
#   dbx_retail.gold.dim_products
#   dbx_retail.gold.dim_customers
#   dbx_retail.gold.dim_vendors
#   dbx_retail.gold.mv_revenue_by_region
#   dbx_retail.gold.mv_return_rate_by_vendor
#   dbx_retail.gold.mv_slow_moving_products
#
# STEP 3 — Add these context instructions to the Genie Space:
#
#   "fact_orders contains one row per order. Each order has a total_sales amount,
#    order_status, ship_mode, and dates. Join to dim_customers on customer_id
#    and to dim_products on product_id for full context.
#
#    fact_returns contains approved and pending return records. refund_amount is
#    the dollar value of the return. return_reason explains why the item was returned.
#
#    mv_slow_moving_products flags products where sales velocity is below the
#    regional average. is_slow_moving = Yes means the product is underperforming.
#
#    mv_return_rate_by_vendor shows the percentage of orders that result in returns
#    for each vendor. A return_rate above 0.20 (20%) is considered high.
#
#    mv_revenue_by_region shows total monthly revenue and order count per region.
#    Use order_month for time-based filtering."
#
# STEP 4 — Demonstrate these 3 natural language queries in Genie:
#   1. "How many products have a return rate above 20% in the West region?"
#   2. "What was the total revenue by region last month?"
#   3. "Which vendors have the highest return rate this quarter?"
#
# =============================================================================